In [1]:
print("✅ All packages installed (or already available)")

✅ All packages installed (or already available)


In [2]:
import nest_asyncio
nest_asyncio.apply()

import os
from crewai import Agent, Task, Crew, Process, LLM

os.environ["CREWAI_OPT_OUT_TELEMETRY"] = "1"
os.environ["OPENAI_API_KEY"] = "dummy"

print("✅ Imports successful")

✅ Imports successful


In [3]:
llm = LLM(
    model="ollama/llama3",
    base_url="http://localhost:11434",
    temperature=0.3,
    is_litellm=True
)

print("✅ LLM configured — Llama 3 via Ollama")


✅ LLM configured — Llama 3 via Ollama


In [4]:
analyst = Agent(
    role="Subject Analyst",
    goal=(
        "Given the subject '{subject}', identify 6–10 key topics a student "
        "must cover to be well-prepared for an exam. Prioritize foundational "
        "topics first, then advanced ones."
    ),
    backstory=(
        "You are an experienced academic tutor who knows exactly which topics "
        "matter most in any subject and how to sequence them logically."
    ),
    tools=[],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

planner = Agent(
    role="Study Planner",
    goal=(
        "Using the topic list from the analyst, create a realistic day-by-day "
        "study plan for {days} days covering the subject '{subject}'. "
        "Each day should have 1–2 topics, a goal, and a tip."
    ),
    backstory=(
        "You are a productivity coach who designs efficient, achievable study "
        "schedules that keep students motivated and on track."
    ),
    tools=[],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("✅ Agents ready: Subject Analyst + Study Planner")


✅ Agents ready: Subject Analyst + Study Planner


In [5]:
def build_tasks(subject, days):
    analysis_task = Task(
        description=(
            f"Subject: {subject}\n"
            f"Days available: {days}\n\n"
            f"List 6–10 key topics a student must study for '{subject}'. "
            f"Order them from foundational to advanced. "
            f"For each topic, write one sentence explaining why it matters."
        ),
        expected_output=(
            "A numbered list of 6–10 topics, each with a one-sentence explanation."
        ),
        agent=analyst
    )

    planning_task = Task(
        description=(
            f"Using the topic list above, build a {days}-day study plan for '{subject}'.\n"
            f"Format each day like this:\n"
            f"**Day N** — Topic(s): <topics> | Goal: <what to achieve> | Tip: <study tip>\n"
            f"Make sure every topic from the list is covered across the {days} days.\n"
            f"Add a motivational closing line at the end."
        ),
        expected_output=(
            f"A {days}-day study plan in the format above, plus a closing motivational line."
        ),
        agent=planner
    )

    return analysis_task, planning_task

print("✅ Task builder ready")


✅ Task builder ready


In [6]:
import asyncio
import threading

def run_study_planner(subject, days):
    print(f"\n{'='*55}")
    print(f"  📚 Generating Study Plan")
    print(f"  Subject : {subject}")
    print(f"  Days    : {days}")
    print(f"{'='*55}\n")

    analysis_task, planning_task = build_tasks(subject, days)

    crew = Crew(
        agents=[analyst, planner],
        tasks=[analysis_task, planning_task],
        process=Process.sequential
    )

    result_container = {}

    def run_in_thread():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            result_container["result"] = crew.kickoff(inputs={"subject": subject, "days": days})
        finally:
            loop.close()

    thread = threading.Thread(target=run_in_thread)
    thread.start()
    thread.join()

    result = result_container.get("result")

    print("\n" + "="*55)
    print("  ✅ STUDY PLAN OUTPUT")
    print("="*55 + "\n")
    print(result)
    return result

print("✅ Runner function ready")

✅ Runner function ready


In [7]:
result1 = run_study_planner(subject="Python Programming", days=7)



  📚 Generating Study Plan
  Subject : Python Programming
  Days    : 7



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Subject Analyst                                                                                         │
│                                                                                                                 │
│  Task: Subject: Python Programming                                                                              │
│  Days available: 7                                                                                              │
│                                                                                                                 │
│  List 6–10 key topics a student must study for 'Python Programming'. Order them from foundational to advanced.  │
│  For each topic, write one sentence explaining why it matters.                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Subject Analyst                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here are the key topics a student must cover to be well-prepared for an exam in Python Programming,            │
│  prioritizing foundational topics first and then advanced ones:                                                 │
│                                                                                                                 │
│  1. **Variables and Data Types**: Understanding how to declare and use variables, as well as the different      │
│  data types available in Python (e.g., strings, lists, dictionaries), is crucial because it lays the            │
│  foundation for all subsequent programming.                                                                     │
│  2. **Basic Syntax and Control Structures**: Familiarity with basic syntax elements such as indentation,        │
│  comments, and print statements, as well as control structures like if-else statements and for loops, is        │
│  essential because these building blocks allow you to write logical and efficient code.                         │
│  3. **Functions**: Mastering the concept of functions, including defining and calling them, is vital because    │
│  it enables you to modularize your code, reuse logic, and make it more readable and maintainable.               │
│  4. **Lists and Tuples**: Understanding how to work with lists (including indexing, slicing, and manipulating)  │
│  and tuples is important because they are fundamental data structures in Python that allow you to store and     │
│  manipulate collections of data.                                                                                │
│  5. **Conditional Statements and Loops**: Being able to write effective conditional statements (if-else) and    │
│  loops (for, while) is critical because these control structures enable you to make decisions and repeat        │
│  actions based on conditions.                                                                                   │
│  6. **Dictionaries and Sets**: Familiarity with dictionaries (key-value pairs) and sets (unordered collections  │
│  of unique elements) is important because they provide additional data structures for storing and manipulating  │
│  complex data.                                                                                                  │
│  7. **Error Handling and Debugging**: Understanding how to handle errors and debug your code using tools like   │
│  try-except blocks, print statements, and the pdb module is essential because it helps you identify and fix     │
│  issues in your code.                                                                                           │
│  8. **Object-Oriented Programming (OOP) Concepts**: Familiarity with OOP concepts such as classes, objects,     │
│  inheritance, polymorphism, and encapsulation is important because they allow you to create reusable and        │
│  modular code that models real-world systems.                                                                   │
│  9. **File Input/Output and Persistence**: Understanding how to read and write files, as well as persist data   │
│  using libraries like pickle or JSON, is crucial because it enables you to store and retrieve data from         │
│  external sources.                                                                                              │
│  10. **Modules and Packages**: Familiarity with importi

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study Planner                                                                                           │
│                                                                                                                 │
│  Task: Using the topic list above, build a 7-day study plan for 'Python Programming'.                           │
│  Format each day like this:                                                                                     │
│  **Day N** — Topic(s): <topics> | Goal: <what to achieve> | Tip: <study tip>                                    │
│  Make sure every topic from the list is covered across the 7 days.                                              │
│  Add a motivational closing line at the end.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study Planner                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Day 1** — Topic(s): Variables and Data Types | Goal: Understand how to declare and use variables in Python,  │
│  as well as the different data types available. | Tip: Start by practicing declaring and using variables with   │
│  different data types, such as strings, lists, and dictionaries.                                                │
│                                                                                                                 │
│  **Day 2** — Topic(s): Basic Syntax and Control Structures | Goal: Familiarize yourself with basic syntax       │
│  elements like indentation, comments, and print statements, as well as control structures like if-else          │
│  statements and for loops. | Tip: Practice writing simple programs using these basic syntax elements and        │
│  control structures to get a feel for how they work.                                                            │
│                                                                                                                 │
│  **Day 3** — Topic(s): Functions | Goal: Master the concept of functions in Python, including defining and      │
│  calling them. | Tip: Start by defining and calling simple functions, then move on to more complex ones that    │
│  take arguments and return values.                                                                              │
│                                                                                                                 │
│  **Day 4** — Topic(s): Lists and Tuples | Goal: Understand how to work with lists (including indexing,          │
│  slicing, and manipulating) and tuples in Python. | Tip: Practice creating, indexing, and manipulating lists    │
│  and tuples to get a feel for their capabilities.                                                               │
│                                                                                                                 │
│  **Day 5** — Topic(s): Conditional Statements and Loops | Goal: Be able to write effective conditional          │
│  statements (if-else) and loops (for, while). | Tip: Start by writing simple programs using if-else statements  │
│  and for loops, then move on to more complex ones that use these control structures.                            │
│                                                                                                                 │
│  **Day 6** — Topic(s): Dictionaries and Sets | Goal: Familiarize yourself with dictionaries (key-value pairs)   │
│  and sets (unordered collections of unique elements). | Tip: Practice creating, indexing, and manipulating      │
│  dictionaries and sets to get a feel for their capabilities.                                                    │
│                                                                                                                 │
│  **Day 7** — Topic(s): Error Handling and Debugging | Goal: Understand how to handle errors and debug your      │
│  code using tools like try-except blocks, print statements, and the pdb module. | Tip: Start by writing simple  │
│  programs that intentionally throw errors, then practice debugging them using these tools.                      │
│                                                                                                                 │
│  Remember, consistency is key! Stick to your study plan


  ✅ STUDY PLAN OUTPUT

**Day 1** — Topic(s): Variables and Data Types | Goal: Understand how to declare and use variables in Python, as well as the different data types available. | Tip: Start by practicing declaring and using variables with different data types, such as strings, lists, and dictionaries.

**Day 2** — Topic(s): Basic Syntax and Control Structures | Goal: Familiarize yourself with basic syntax elements like indentation, comments, and print statements, as well as control structures like if-else statements and for loops. | Tip: Practice writing simple programs using these basic syntax elements and control structures to get a feel for how they work.

**Day 3** — Topic(s): Functions | Goal: Master the concept of functions in Python, including defining and calling them. | Tip: Start by defining and calling simple functions, then move on to more complex ones that take arguments and return values.

**Day 4** — Topic(s): Lists and Tuples | Goal: Understand how to work with list

In [8]:
result2 = run_study_planner(subject="Machine Learning", days=14)



  📚 Generating Study Plan
  Subject : Machine Learning
  Days    : 14



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Subject Analyst                                                                                         │
│                                                                                                                 │
│  Task: Subject: Machine Learning                                                                                │
│  Days available: 14                                                                                             │
│                                                                                                                 │
│  List 6–10 key topics a student must study for 'Machine Learning'. Order them from foundational to advanced.    │
│  For each topic, write one sentence explaining why it matters.                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Subject Analyst                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on my expertise as a Subject Analyst, I've identified the key topics in Machine Learning that a student  │
│  must cover to be well-prepared for an exam. Here are the foundational and advanced topics, prioritized         │
│  accordingly:                                                                                                   │
│                                                                                                                 │
│  1. **Introduction to Machine Learning**: Understanding the basics of machine learning, including its           │
│  definition, types, and applications, is crucial as it sets the stage for further exploration.                  │
│  2. **Supervised Learning**: Mastering supervised learning concepts, such as regression, classification, and    │
│  clustering, provides a solid foundation for understanding how machines learn from labeled data.                │
│  3. **Unsupervised Learning**: Familiarity with unsupervised learning techniques like k-means clustering,       │
│  hierarchical clustering, and dimensionality reduction is essential for identifying patterns in unlabeled       │
│  data.                                                                                                          │
│  4. **Linear Algebra**: A strong grasp of linear algebra concepts, including vector spaces, matrices, and       │
│  eigenvectors, is vital for understanding many machine learning algorithms and mathematical derivations.        │
│  5. **Probability Theory**: Understanding probability theory, including Bayes' theorem, conditional             │
│  probability, and independence, enables students to work with uncertainty and make informed decisions in        │
│  machine learning models.                                                                                       │
│  6. **Model Evaluation Metrics**: Knowing how to evaluate model performance using metrics like accuracy,        │
│  precision, recall, F1-score, and mean squared error is critical for selecting the best-performing models.      │
│  7. **Regularization Techniques**: Familiarity with regularization techniques, such as L1 and L2                │
│  regularization, dropout, and early stopping, helps prevent overfitting and improves model generalizability.    │
│  8. **Neural Networks**: Understanding neural networks, including their architecture, activation functions,     │
│  and backpropagation algorithm, is essential for working with deep learning models.                             │
│  9. **Gradient Descent and Optimization Algorithms**: Knowledge of gradient descent and optimization            │
│  algorithms like stochastic gradient descent, Adam, and RMSProp enables students to train complex models        │
│  efficiently.                                                                                                   │
│  10. **Hyperparameter Tuning and Model Selection**: Understanding how to tune hyperparameters and select the    │
│  best-performing model using techniques like cross-validation and grid search is crucial for achieving optimal  │
│  performance.                                                                                                   │
│                                                                                                                 │
│  By covering these foundational and advanced topics in 

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study Planner                                                                                           │
│                                                                                                                 │
│  Task: Using the topic list above, build a 14-day study plan for 'Machine Learning'.                            │
│  Format each day like this:                                                                                     │
│  **Day N** — Topic(s): <topics> | Goal: <what to achieve> | Tip: <study tip>                                    │
│  Make sure every topic from the list is covered across the 14 days.                                             │
│  Add a motivational closing line at the end.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study Planner                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Day 1** — Topic(s): Introduction to Machine Learning | Goal: Understand the definition, types, and           │
│  applications of machine learning | Tip: Start by reading the introduction chapter in a relevant textbook or    │
│  online resource to get a solid grasp of the basics.                                                            │
│                                                                                                                 │
│  **Day 2** — Topic(s): Supervised Learning (Regression) | Goal: Learn about regression algorithms and their     │
│  applications | Tip: Focus on understanding the mathematical derivations behind popular regression techniques   │
│  like linear regression and logistic regression.                                                                │
│                                                                                                                 │
│  **Day 3** — Topic(s): Unsupervised Learning (Clustering) | Goal: Familiarize yourself with k-means clustering  │
│  and hierarchical clustering | Tip: Practice implementing these algorithms using a programming language of      │
│  your choice to solidify your understanding.                                                                    │
│                                                                                                                 │
│  **Day 4** — Topic(s): Linear Algebra (Vector Spaces) | Goal: Understand the basics of vector spaces,           │
│  matrices, and eigenvectors | Tip: Review linear algebra concepts from previous studies or online resources to  │
│  ensure a strong foundation.                                                                                    │
│                                                                                                                 │
│  **Day 5** — Topic(s): Probability Theory (Bayes' Theorem) | Goal: Learn about conditional probability,         │
│  independence, and Bayes' theorem | Tip: Practice solving problems involving conditional probability and        │
│  Bayes' theorem to build your problem-solving skills.                                                           │
│                                                                                                                 │
│  **Day 6** — Topic(s): Model Evaluation Metrics | Goal: Understand how to evaluate model performance using      │
│  accuracy, precision, recall, F1-score, and mean squared error | Tip: Focus on understanding the mathematical   │
│  derivations behind these metrics and practice calculating them for different scenarios.                        │
│                                                                                                                 │
│  **Day 7** — Topic(s): Regularization Techniques (L1 and L2) | Goal: Learn about regularization techniques to   │
│  prevent overfitting | Tip: Review the concepts of bias-variance tradeoff and how regularization techniques     │
│  help achieve a balance between the two.                                                                        │
│                                                                                                                 │
│  **Day 8** — Topic(s): Neural Networks (Architecture) | Goal: Understand the basic architecture of neural       │
│  networks, including activation functions and backpropa


  ✅ STUDY PLAN OUTPUT

**Day 1** — Topic(s): Introduction to Machine Learning | Goal: Understand the definition, types, and applications of machine learning | Tip: Start by reading the introduction chapter in a relevant textbook or online resource to get a solid grasp of the basics.

**Day 2** — Topic(s): Supervised Learning (Regression) | Goal: Learn about regression algorithms and their applications | Tip: Focus on understanding the mathematical derivations behind popular regression techniques like linear regression and logistic regression.

**Day 3** — Topic(s): Unsupervised Learning (Clustering) | Goal: Familiarize yourself with k-means clustering and hierarchical clustering | Tip: Practice implementing these algorithms using a programming language of your choice to solidify your understanding.

**Day 4** — Topic(s): Linear Algebra (Vector Spaces) | Goal: Understand the basics of vector spaces, matrices, and eigenvectors | Tip: Review linear algebra concepts from previous studies 

In [ ]:
# 🔧 Customize here
my_subject = "Data Structures and Algorithms"
my_days    = 10

result3 = run_study_planner(subject=my_subject, days=my_days)



  📚 Generating Study Plan
  Subject : Data Structures and Algorithms
  Days    : 10



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Subject Analyst                                                                                         │
│                                                                                                                 │
│  Task: Subject: Data Structures and Algorithms                                                                  │
│  Days available: 10                                                                                             │
│                                                                                                                 │
│  List 6–10 key topics a student must study for 'Data Structures and Algorithms'. Order them from foundational   │
│  to advanced. For each topic, write one sentence explaining why it matters.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Subject Analyst                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here are the key topics to study in Data Structures and Algorithms, prioritized from foundational to           │
│  advanced:                                                                                                      │
│                                                                                                                 │
│  1. **Arrays**: Understanding arrays is crucial because it lays the foundation for more complex data            │
│  structures and algorithms.                                                                                     │
│  2. **Linked Lists**: Mastering linked lists is essential as they are a fundamental building block for many     │
│  other data structures and algorithms.                                                                          │
│  3. **Stacks and Queues**: Familiarity with stacks and queues is vital because they provide a solid             │
│  understanding of basic algorithmic concepts like LIFO (Last In, First Out) and FIFO (First In, First Out).     │
│  4. **Trees**: Studying trees is important because they are a fundamental data structure used in many           │
│  algorithms, such as traversals, searches, and sorts.                                                           │
│  5. **Graphs**: Understanding graphs is critical because they are used to model complex relationships between   │
│  objects and are essential for many real-world applications.                                                    │
│  6. **Hash Tables**: Familiarity with hash tables is important because they provide a fast and efficient way    │
│  to store and retrieve data, making them a crucial component in many algorithms.                                │
│  7. **Sorting Algorithms**: Mastering sorting algorithms like Bubble Sort, Selection Sort, Insertion Sort,      │
│  Merge Sort, Quick Sort, and Heap Sort is vital because they are used extensively in many applications.         │
│  8. **Searching Algorithms**: Understanding searching algorithms like Linear Search, Binary Search, and Hash    │
│  Table Search is important because they provide efficient ways to find specific data within a dataset.          │
│  9. **Dynamic Programming**: Studying dynamic programming is crucial because it allows for solving complex      │
│  problems by breaking them down into smaller subproblems and solving each one recursively or iteratively.       │
│  10. **Greedy Algorithms**: Familiarity with greedy algorithms like Huffman Coding, Activity Selection          │
│  Problem, and Coin Changing Problem is important because they provide efficient solutions to certain types of   │
│  optimization problems.                                                                                         │
│                                                                                                                 │
│  These topics will provide a solid foundation for understanding data structures and algorithms, allowing        │
│  students to tackle more complex problems and prepare them well for an exam.                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study Planner                                                                                           │
│                                                                                                                 │
│  Task: Using the topic list above, build a 10-day study plan for 'Data Structures and Algorithms'.              │
│  Format each day like this:                                                                                     │
│  **Day N** — Topic(s): <topics> | Goal: <what to achieve> | Tip: <study tip>                                    │
│  Make sure every topic from the list is covered across the 10 days.                                             │
│  Add a motivational closing line at the end.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 📝 Summary & Reflection

### What this agent does
| Component | Role |
|---|---|
| `Subject Analyst` Agent | Breaks subject into prioritized topics |
| `Study Planner` Agent | Converts topics into a day-by-day schedule |
| `Crew` (Sequential) | Passes analyst output → planner automatically |
| `Llama 3 via Ollama` | Powers both agents locally, no internet needed |

### Key Concepts Used
- **Multi-agent collaboration** — two specialized agents, one goal
- **Sequential process** — output of Task 1 feeds into Task 2
- **Local LLM** — no API cost, full privacy, runs on your machine
- **Dynamic inputs** — same crew handles any subject and any number of days

### What I learned
Building this agent showed how breaking a complex task (exam prep) into smaller
roles (analyze → plan) makes AI output more structured and useful than asking
a single model everything at once.
